In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score

# -------------------------------------------------
# Load Dataset
# -------------------------------------------------

cols = [
    'age',
    'workclass',
    'fnlwgt',
    'education',
    'education-num',
    'marital-status',
    'occupation',
    'relationship',
    'race',
    'sex',
    'capital-gain',
    'capital-loss',
    'hours-per-week',
    'native-country',
    'income'
]

df = pd.read_csv("adult.csv", names=cols)

print(df.head())


# -------------------------------------------------
# m. Data Cleaning
# -------------------------------------------------

# Replace ? with NaN
df.replace('?', np.nan, inplace=True)

# Remove missing values
df.dropna(inplace=True)

# Remove negative values from numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df = df[df[col] >= 0]

print("\nCleaned Data:")
print(df.head())


# -------------------------------------------------
# n. Error Correcting (Outlier Detection)
# -------------------------------------------------

# IQR Method (only numeric feature columns)

feature_cols = [
    'age',
    'fnlwgt',
    'education-num',
    'capital-gain',
    'capital-loss',
    'hours-per-week'
]

for col in feature_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

print("\nData after Outlier Removal:")
print(df.head())


# -------------------------------------------------
# o. Data Transformation
# -------------------------------------------------

# Encode categorical columns
le = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le.fit_transform(df[col])

# Features and Target
X = df.drop('income', axis=1)
y = df['income']

# Standardization
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


# -------------------------------------------------
# p. Build Models
# -------------------------------------------------

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

# Logistic Regression
lr_model = LogisticRegression()

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, lr_pred)

print("\nLogistic Regression Accuracy:", lr_accuracy)


# Naive Bayes
nb_model = GaussianNB()

nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)

nb_accuracy = accuracy_score(y_test, nb_pred)

print("\nNaive Bayes Accuracy:", nb_accuracy)


# Compare Accuracy
if lr_accuracy > nb_accuracy:
    print("\nLogistic Regression performs better.")
else:
    print("\nNaive Bayes performs better.")

   age          workclass    fnlwgt   education  education-num  \
0   39          State-gov   77516.0   Bachelors           13.0   
1   50   Self-emp-not-inc   83311.0   Bachelors           13.0   
2   38            Private  215646.0     HS-grad            9.0   
3   53            Private  234721.0        11th            7.0   
4   28            Private  338409.0   Bachelors           13.0   

        marital-status          occupation    relationship    race      sex  \
0        Never-married        Adm-clerical   Not-in-family   White     Male   
1   Married-civ-spouse     Exec-managerial         Husband   White     Male   
2             Divorced   Handlers-cleaners   Not-in-family   White     Male   
3   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
4   Married-civ-spouse      Prof-specialty            Wife   Black   Female   

   capital-gain  capital-loss  hours-per-week  native-country  income  
0        2174.0           0.0            40.0   United-S